In [1]:
import pandas as pd
import altair as alt

# Disable max rows limit (needed for large datasets)
alt.data_transformers.disable_max_rows()

# Load the dataset directly from URL
url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv"
df = pd.read_csv(url)

# Quick look at the data
print(df.shape)
df.head()

(4747, 29)


,observed,location_details,county,state,season,title,latitude,longitude,date,number,...,precip_intensity,precip_probability,precip_type,pressure,summary,uv_index,visibility,wind_bearing,wind_speed,location
0,Ed L. was salmon fishing with a companion in P...,East side of Prince William Sound,Valdez-Chitina-Whittier County,Alaska,Fall,NaN,NaN,NaN,NaN,1261.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,heh i kinda feel a little dumb that im reporti...,"the road is off us rt 80, i dont know the exit...",Warren County,New Jersey,Fall,NaN,NaN,NaN,NaN,438.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,I was on my way to Claremont from Lebanon on R...,Close to Claremont down 120 not far from Kings...,Sullivan County,New Hampshire,Summer,Report 55269: Dawn sighting at Stevens Brook o...,43.41549,-72.33093,2016-06-07,55269.0,...,0.001,0.7,rain,998.87,Mostly cloudy throughout the day.,6.0,9.70,262.0,0.49,POINT(-72.33093000000001 43.415490000000005)
3,I was northeast of Macy Nebraska along the Mis...,Latitude & Longitude : 42.158230 -96.344197,Thurston County,Nebraska,Spring,Report 59757: Possible daylight sighting of a ...,42.15685,-96.34203,2018-05-25,59757.0,...,0.000,0.0,NaN,1008.07,Partly cloudy in the morning.,10.0,8.25,193.0,3.33,POINT(-96.34203000000001 42.15685)
4,"While this incident occurred a long time ago, ...","Ward County, Just outside of a the Minuteman T...",Ward County,North Dakota,Spring,Report 751: Hunter describes described being s...,48.25422,-101.31660,2000-04-21,751.0,...,NaN,NaN,rain,1011.47,Partly cloudy until evening.,6.0,10.00,237.0,11.14,POINT(-101.3166 48.254220000000004)


In [2]:
print(df.columns.tolist())
print(df.dtypes)
print(df.isnull().sum())

['observed', 'location_details', 'county', 'state', 'season', 'title', 'latitude', 'longitude', 'date', 'number', 'classification', 'geohash', 'temperature_high', 'temperature_mid', 'temperature_low', 'dew_point', 'humidity', 'cloud_cover', 'moon_phase', 'precip_intensity', 'precip_probability', 'precip_type', 'pressure', 'summary', 'uv_index', 'visibility', 'wind_bearing', 'wind_speed', 'location']
observed               object
location_details       object
county                 object
state                  object
season                 object
title                  object
latitude              float64
longitude             float64
date                   object
number                float64
classification         object
geohash                object
temperature_high      float64
temperature_mid       float64
temperature_low       float64
dew_point             float64
humidity              float64
cloud_cover           float64
moon_phase            float64
precip_intensity      float

In [3]:
# Count sightings per state and take top 20
state_counts = (
    df['state']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'state', 'state': 'count'})
    .head(20)
)

# Rename columns clearly
state_counts.columns = ['state', 'count']

plot1 = alt.Chart(state_counts).mark_bar().encode(
    x=alt.X('count:Q', title='Number of Sightings'),
    y=alt.Y('state:N', sort='-x', title='State'),
    color=alt.Color(
        'count:Q',
        scale=alt.Scale(scheme='teals'),
        title='Sightings'
    ),
    tooltip=['state:N', 'count:Q']
).properties(
    title='Top 20 States by Bigfoot Sightings',
    width=500,
    height=400
)

plot1

alt.Chart(...)

In [8]:
# Use URL reference instead of embedding all data inline
df_clean_url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv"

season_dropdown = alt.binding_select(
    options=valid_seasons,
    name='Select Season: '
)
season_selection = alt.selection_point(
    fields=['season'],
    bind=season_dropdown,
    value='Summer'
)

plot2 = alt.Chart(df_clean_url).mark_circle(size=40, opacity=0.6).transform_filter(
    "datum.season == 'Spring' || datum.season == 'Summer' || datum.season == 'Fall' || datum.season == 'Winter'"
).transform_filter(
    "datum.latitude >= 24 && datum.latitude <= 50 && datum.longitude >= -125 && datum.longitude <= -66"
).encode(
    x=alt.X('longitude:Q', title='Longitude', scale=alt.Scale(domain=[-130, -60])),
    y=alt.Y('latitude:Q', title='Latitude', scale=alt.Scale(domain=[25, 55])),
    color=alt.Color(
        'classification:N',
        scale=alt.Scale(scheme='set1'),
        title='Classification'
    ),
    tooltip=['state:N', 'season:N', 'classification:N', 'latitude:Q', 'longitude:Q']
).add_params(
    season_selection
).transform_filter(
    season_selection
).properties(
    title='Bigfoot Sightings by Location, Season & Classification',
    width=600,
    height=400
)

plot2

alt.Chart(...)

In [6]:
plot1.save('plot1.json')
plot2.save('plot2.json')